# 01 — Synthetic history and exploratory analysis
Phase 3 artifact review. Read the frozen eight-week history; do not regenerate it. All observations are **synthetic**, not Wokwi measurements.
Run all cells in order using the approved local Smart Parking ML interpreter. This notebook writes no datasets, models or reports.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.generate_data import SLOTS, read_raw, content_digest
from ml.prepare_data import file_hash
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
print('Interpreter:', sys.executable)
print('Repository:', ROOT)

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe
Repository: C:\Users\vighn\OneDrive\Desktop\iot project


## Provenance and assumptions
The frozen generator uses seed 42, Poisson arrivals, daily demand variation and lognormal dwell times. Weekday commute/midday peaks differ from weekend leisure peaks. Capacity is 12; excess demand is lost. Initial state is vacant; stays continue across midnight.
There is no site calibration, weather, holiday model, sensor noise or missingness. Arrival/departure transitions demonstrate simulator behavior, not empirical realism.

In [2]:
record = manifest['raw_file']
assert file_hash(record['path']) == record['sha256']
raw = read_raw(record['path'])
assert len(raw) == 56 * 24 * 60 == manifest['raw_rows']
assert raw.timestamp.is_unique and raw.timestamp.is_monotonic_increasing
assert raw.timestamp.diff().iloc[1:].eq(pd.Timedelta(minutes=1)).all()
assert not raw.isna().any().any()
assert raw.data_source.eq('synthetic').all()
assert list(raw.loc[:, list(SLOTS)].columns) == [f'P{i}' for i in range(1, 13)]
assert raw[list(SLOTS)].isin([0, 1]).all().all()
assert raw.total_occupied.eq(raw[list(SLOTS)].sum(axis=1)).all()
assert content_digest(raw) == manifest['raw_content_digest']
print('PASS: checksum, 80,640 minute rows, 12 binary slots, provenance and slot sums')
display(raw.head(3))
display(pd.Series(manifest['assumptions'], name='Frozen assumptions'))

PASS: checksum, 80,640 minute rows, 12 binary slots, provenance and slot sums


,timestamp,data_source,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,total_occupied,arrivals,departures
0,2026-09-14 00:00:00+05:30,synthetic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2026-09-14 00:01:00+05:30,synthetic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2026-09-14 00:02:00+05:30,synthetic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


slot_state       0=VACANT,1=OCCUPIED; perfect synthetic observa...
weekday          Mon-Fri commute and midday peaks; Sat-Sun leis...
duration         lognormal sigma=0.5; weekday morning mean=300 ...
demand           Poisson arrivals with Gaussian hour-of-day pea...
capacity         12; excess arrivals are lost, not queued; no r...
initial_state    all vacant at midnight; no daily state reset; ...
calendar         fixed 2026 calendar; no holiday, weather or re...
Name: Frozen assumptions, dtype: str

## Slot transitions and daily/weekly patterns
Compute arrivals as 0→1 and departures as 1→0 using signed differences. The first row has no preceding observation and is excluded from differences. Summaries below are descriptive, not targets or fitted preprocessing.

In [3]:
changes = raw[list(SLOTS)].astype('int16').diff().fillna(0)
arrivals = changes.eq(1).sum(axis=1)
departures = changes.eq(-1).sum(axis=1)
assert arrivals.eq(raw.arrivals).all() and departures.eq(raw.departures).all()
transitions = pd.DataFrame({'arrivals': changes.eq(1).sum(), 'departures': changes.eq(-1).sum(), 'occupied_fraction': raw[list(SLOTS)].mean()})
assert transitions[['arrivals', 'departures']].gt(0).all().all()
assert raw.total_occupied.astype('int16').diff().iloc[1:].eq((arrivals - departures).iloc[1:]).all()
display(transitions)
summary = raw.assign(hour=raw.timestamp.dt.hour, weekend=raw.timestamp.dt.dayofweek.ge(5))
display(summary.groupby('weekend').agg(mean_occupied=('total_occupied', 'mean'), arrivals=('arrivals', 'sum'), departures=('departures', 'sum')))
display(summary.groupby('hour').total_occupied.mean().round(3).rename('mean_occupied').to_frame())
print('PASS: all slots have arrivals/departures; occupancy changes reconcile')

,arrivals,departures,occupied_fraction
P1,100,100,0.224281
P2,93,93,0.208569
P3,91,91,0.203621
P4,95,95,0.189943
P5,84,84,0.176500
P6,84,84,0.175025
P7,86,86,0.201563
P8,89,89,0.210218
P9,82,82,0.195871
P10,82,82,0.199417


C:\Users\vighn\AppData\Local\Temp\ipykernel_32604\2425367739.py:9: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  summary = raw.assign(hour=raw.timestamp.dt.hour, weekend=raw.timestamp.dt.dayofweek.ge(5))


,mean_occupied,arrivals,departures
weekend,,,
False,2.903490,824,824
True,0.972569,218,218


,mean_occupied
hour,
0,0.264
1,0.197
2,0.185
3,0.126
4,0.131
5,0.143
6,0.419
7,1.390
8,3.333


PASS: all slots have arrivals/departures; occupancy changes reconcile


In [4]:
storage = {p.as_posix(): p.stat().st_size for folder in ('ml/data', 'ml/artifacts') for p in Path(folder).rglob('*') if p.is_file()}
assert sum(storage.values()) < 50_000_000
display(pd.Series(storage, name='bytes').to_frame())
print('Data/model bytes:', sum(storage.values()))
print('PASS: existing history reviewed without regeneration or artifact writes')

,bytes
ml/data/.gitkeep,0
ml/data/phase3_dataset_manifest.json,4489
ml/data/synthetic_parking_8weeks_ml.csv.gz,2077563
ml/data/synthetic_parking_8weeks_raw.csv,5245712
ml/artifacts/.gitkeep,0
ml/artifacts/phase4/.gitattributes,8
ml/artifacts/phase4/model_results.json,4854
ml/artifacts/phase4/parking_models.joblib,851974
ml/artifacts/phase4/validation_selection.json,5519


Data/model bytes: 8190119
PASS: existing history reviewed without regeneration or artifact writes


## Continuation
Notebook 02 verifies causal features, future targets and chronological purges. Consult `ml/README.md` and the frozen Phase 3 PDF for the original generation command and validation evidence. This notebook intentionally does not run that generation command.